
# K-Means Tutorial Demo for a Session

This notebook is a **tutorial-style demo** of the **k-means algorithm**, with the mathematics written explicitly in display form so it can be used in class.

## Roadmap

1. Mathematical setup of clustering
2. The k-means objective and why the centroid is the mean
3. Lloyd's algorithm, step by step
4. Demo on an easy dataset
5. Exercises with solutions
6. Choosing the number of clusters $K$
7. A failure case where k-means struggles
8. A practical application: image color quantization



> **Use in class:** this notebook is written in tutorial style, with the main formulas in display math using `$$...$$`. Run cells top to bottom to regenerate all figures live during the session.



## 1. Mathematical setup

We are given data points

$$
X = \{x_1, x_2, \dots, x_n\}, \qquad x_i \in \mathbb{R}^d.
$$

In clustering, there are **no labels**. We want to divide the data into $K$ groups.
A clustering can be described by assignment variables

$$
z_i \in \{1,2,\dots,K\},
$$

where $z_i = k$ means that point $x_i$ belongs to cluster $k$.

For each cluster $k$, we also introduce a centroid

$$
\mu_k \in \mathbb{R}^d.
$$

The central idea of k-means is simple:

- each point is assigned to one cluster,
- each cluster is represented by its centroid,
- good clusters are those where points stay close to their assigned centroid.



## 2. The k-means objective

The k-means objective is the **within-cluster sum of squares**:

$$
J(\{z_i\}, \{\mu_k\})
=
\sum_{i=1}^{n} \left\|x_i - \mu_{z_i}\right\|_2^2.
$$

Equivalently, if $C_k = \{x_i : z_i = k\}$ is the set of points in cluster $k$, then

$$
J = \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|_2^2.
$$

So k-means solves the optimization problem

$$
\min_{\{z_i\}, \{\mu_k\}} \sum_{i=1}^{n} \left\|x_i - \mu_{z_i}\right\|_2^2.
$$

This objective rewards clusters that are **compact**.



## 3. Why is the centroid the mean?

Suppose the assignments are fixed, and we focus on one cluster $C_k$.
We want to choose $\mu_k$ to minimize

$$
f(\mu_k) = \sum_{x_i \in C_k} \|x_i - \mu_k\|_2^2.
$$

Expand and differentiate with respect to $\mu_k$:

$$
\nabla_{\mu_k} f(\mu_k)
= -2 \sum_{x_i \in C_k} (x_i - \mu_k).
$$

Setting the gradient equal to zero gives

$$
\sum_{x_i \in C_k} (x_i - \mu_k) = 0.
$$

Hence

$$
|C_k| \, \mu_k = \sum_{x_i \in C_k} x_i,
$$

so the optimal centroid is

$$
\mu_k = \frac{1}{|C_k|} \sum_{x_i \in C_k} x_i.
$$

That is the reason the method is called **k-means**.



## 4. Lloyd's algorithm

Directly solving the joint optimization over assignments and centroids is hard, but if one side is fixed, the other is easy.

### Assignment step
Given centroids $\mu_1, \dots, \mu_K$, assign each point to the nearest centroid:

$$
z_i = \arg\min_{k \in \{1,\dots,K\}} \|x_i - \mu_k\|_2^2.
$$

### Update step
Given assignments, recompute each centroid as the mean of its assigned points:

$$
\mu_k = \frac{1}{|C_k|} \sum_{x_i \in C_k} x_i.
$$

These two steps are repeated until the assignments stop changing or the objective stops decreasing.

### Key fact
Each Lloyd step does **not increase** the objective:

$$
J^{(t+1)} \leq J^{(t)}.
$$

So the algorithm converges to a **local minimum**, not necessarily the global minimum.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs, make_moons, load_sample_image
from sklearn.metrics import silhouette_score

np.random.seed(7)
plt.rcParams["figure.figsize"] = (6, 5)


def plot_clusters(X, labels=None, centers=None, title="", ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    if labels is None:
        ax.scatter(X[:, 0], X[:, 1], s=35)
    else:
        ax.scatter(X[:, 0], X[:, 1], c=labels, s=35)
    if centers is not None:
        ax.scatter(centers[:, 0], centers[:, 1], marker="X", s=260)
    ax.set_title(title)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    return ax



## 5. Demo 1: k-means on well-separated blobs

This is the setting where k-means shines.
The clusters are roughly compact and spherical, which matches the geometry built into the squared Euclidean objective.


In [ ]:

X_blobs, y_blobs = make_blobs(
    n_samples=450,
    centers=[(-4, -1), (0, 4), (4, -1)],
    cluster_std=[0.9, 1.1, 0.8],
    random_state=12,
)

kmeans3 = KMeans(n_clusters=3, init="k-means++", n_init=20, random_state=12)
labels3 = kmeans3.fit_predict(X_blobs)
centers3 = kmeans3.cluster_centers_

fig, ax = plt.subplots()
plot_clusters(X_blobs, labels3, centers3, title="K-means on well-separated blobs", ax=ax)
plt.show()

print("Inertia =", round(kmeans3.inertia_, 2))



For this dataset, the clusters found by k-means line up well with the visible groups.
The quantity reported as **inertia** is exactly the optimized objective

$$
\text{Inertia} = \sum_{i=1}^{n} \|x_i - \mu_{z_i}\|_2^2.
$$



## 6. Exercise 1: one Lloyd iteration by hand

Consider the points

$$
X = \{(1,1), (1.5,2), (3,4), (5,7), (3.5,5), (4.5,5), (3.5,4.5)\}
$$

and the initial centroids

$$
\mu_1^{(0)} = (1,2), \qquad \mu_2^{(0)} = (5,5).
$$

### Task

1. For each point, compute the squared distances to the two centroids.
2. Assign each point to the nearest centroid.
3. Recompute the two centroids using the mean formula.


In [ ]:

X_small = np.array([
    [1.0, 1.0],
    [1.5, 2.0],
    [3.0, 4.0],
    [5.0, 7.0],
    [3.5, 5.0],
    [4.5, 5.0],
    [3.5, 4.5],
])

centers0 = np.array([
    [1.0, 2.0],
    [5.0, 5.0],
])

fig, ax = plt.subplots()
plot_clusters(X_small, centers=centers0, title="Exercise 1: data and initial centroids", ax=ax)
plt.show()

print("Initial centroids:")
print(centers0)


In [ ]:

# Student work area
# Fill in the labels and updated centroids manually or by code.

def assign_points(X, centers):
    dists = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    labels = dists.argmin(axis=1)
    return dists, labels


dists0, labels1 = assign_points(X_small, centers0)
print("Squared distance matrix:")
print(np.round(dists0, 2))
print() 
print("Nearest-centroid labels:")
print(labels1)



### Solution to Exercise 1

Once the assignments are known, the updated centroids are

$$
\mu_k^{(1)} = \frac{1}{|C_k|} \sum_{x_i \in C_k} x_i.
$$


In [ ]:

new_centers = np.vstack([
    X_small[labels1 == k].mean(axis=0) for k in range(2)
])

print("Updated centroids after one Lloyd step:")
print(np.round(new_centers, 3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_clusters(X_small, None, centers0, title="Initial centroids", ax=axes[0])
plot_clusters(X_small, labels1, new_centers, title="After one Lloyd step", ax=axes[1])
plt.tight_layout()
plt.show()



## 7. Choosing the number of clusters $K$

Two common quantities are:

### Inertia

$$
I(K) = \sum_{i=1}^{n} \|x_i - \mu_{z_i}\|_2^2.
$$

As $K$ increases, inertia always decreases, because the model becomes more flexible.
So inertia alone usually does **not** determine the best $K$.

### Silhouette score
For a point $x_i$, let

$$
a_i = \text{average distance from } x_i \text{ to points in its own cluster},
$$

and let

$$
b_i = \min_{\ell \neq z_i} \text{average distance from } x_i \text{ to cluster } \ell.
$$

The silhouette value of point $i$ is

$$
s_i = \frac{b_i - a_i}{\max(a_i, b_i)}.
$$

The overall silhouette score is the average

$$
S = \frac{1}{n} \sum_{i=1}^{n} s_i.
$$

Interpretation:

- $s_i \approx 1$: well-clustered
- $s_i \approx 0$: near a boundary
- $s_i < 0$: likely misclustered



## 8. Exercise 2: compare inertia and silhouette for different $K$

We will fit k-means for $K = 2,3,4,5,6$ and compare the two scores.
Your job is to look for an **elbow** in inertia and a **peak** in silhouette.


In [ ]:

Ks = range(2, 7)
inertias = []
silhouettes = []
models = {}

for K in Ks:
    km = KMeans(n_clusters=K, init="k-means++", n_init=20, random_state=12)
    labels = km.fit_predict(X_blobs)
    models[K] = (km, labels)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_blobs, labels))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(list(Ks), inertias, marker="o")
axes[0].set_title("Elbow plot")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")

axes[1].plot(list(Ks), silhouettes, marker="o")
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Average silhouette")

plt.tight_layout()
plt.show()

for K, inertia, sil in zip(Ks, inertias, silhouettes):
    print(f"K={K}: inertia={inertia:.2f}, silhouette={sil:.3f}")



### Discussion

For this dataset, the natural answer should be close to $K=3$, because that is the visible number of compact groups.

Mathematically, this is a good moment to remember that k-means is solving

$$
\min_{\{z_i\}, \{\mu_k\}} \sum_{i=1}^{n} \|x_i - \mu_{z_i}\|_2^2,
$$

not “find the true number of clusters” in some universal sense.
The choice of $K$ is model selection layered on top of the objective.



## 9. Failure case: two moons

K-means partitions space into Voronoi regions around centroids, so it naturally prefers convex, roughly spherical clusters.
For two curved moon-shaped clusters, that geometry is wrong.


In [ ]:

X_moons, y_moons = make_moons(n_samples=500, noise=0.08, random_state=5)
km_moons = KMeans(n_clusters=2, init="k-means++", n_init=20, random_state=5)
labels_moons = km_moons.fit_predict(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, s=20)
axes[0].set_title("Underlying moon structure")
axes[0].set_xlabel("x1")
axes[0].set_ylabel("x2")

plot_clusters(X_moons, labels_moons, km_moons.cluster_centers_, title="K-means result on two moons", ax=axes[1])
plt.tight_layout()
plt.show()



The lesson is geometric:

- k-means works well when clusters are compact around centroids,
- k-means struggles when clusters are non-convex or strongly density-based.

That is why methods such as spectral clustering or DBSCAN are often used for shapes like moons, spirals, or nested rings.



## 10. Application: image color quantization

Here each pixel is a 3-dimensional vector

$$
x_i = (R_i, G_i, B_i) \in \mathbb{R}^3.
$$

Running k-means on these vectors finds $K$ representative colors

$$
\mu_1, \dots, \mu_K \in \mathbb{R}^3.
$$

Then each pixel is replaced by the nearest centroid color.
This is a form of **vector quantization**.


In [ ]:

china = load_sample_image("china.jpg")
img = np.array(china, dtype=np.float64) / 255.0
h, w, c = img.shape
pixels = img.reshape(-1, 3)

K = 16
km_img = KMeans(n_clusters=K, init="k-means++", n_init=10, random_state=3)
labels_img = km_img.fit_predict(pixels)
palette = km_img.cluster_centers_
compressed = palette[labels_img].reshape(h, w, c)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img)
axes[0].set_title("Original image")
axes[0].axis("off")

axes[1].imshow(np.clip(compressed, 0, 1))
axes[1].set_title(f"K-means color quantization (K={K})")
axes[1].axis("off")

plt.tight_layout()
plt.show()



## 11. Tutorial summary

The core formulas to remember are these:

### Objective

$$
J = \sum_{i=1}^{n} \|x_i - \mu_{z_i}\|_2^2.
$$

### Assignment step

$$
z_i = \arg\min_k \|x_i - \mu_k\|_2^2.
$$

### Update step

$$
\mu_k = \frac{1}{|C_k|} \sum_{x_i \in C_k} x_i.
$$

### Silhouette score

$$
s_i = \frac{b_i - a_i}{\max(a_i, b_i)}.
$$

If you remember these four equations and the geometry behind them, you understand the backbone of k-means.
